In [1]:
!pip install langdetect sentencepiece -q


ERROR: Could not find a version that satisfies the requirement langdetect (from versions: none)
ERROR: No matching distribution found for langdetect


In [2]:
import os, re
import numpy as np
import pandas as pd
import cv2
import pydicom
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import albumentations as A

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TRAIN_CSV = f"{DATA_DIR}/train.csv"
SERIES_CSV = f"{DATA_DIR}/train_series.csv"
TRAIN_IMG_DIR = f"{DATA_DIR}/train_series"
TEST_CSV = f"{DATA_DIR}/test.csv"
TEST_SERIES_CSV = f"{DATA_DIR}/test_series.csv"
TEST_IMG_DIR = f"{DATA_DIR}/test_series"

MODEL_DIR = "/kaggle/working/models"
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

IMG_SIZE = 224
NUM_SLICES = 3
BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

train = pd.read_csv(TRAIN_CSV)
series_df = pd.read_csv(SERIES_CSV)
gold = train[train[LABELS].notna().all(axis=1)].copy()

print("Config and data loaded. gold shape:", gold.shape)

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


Using device: cuda
Config and data loaded. gold shape: (58, 14)


In [3]:
def list_series_files_sorted(series_folder):
    files = [f for f in os.listdir(series_folder) if f.endswith(".dcm")]
    full_paths = [os.path.join(series_folder, f) for f in files]
    def get_instance_number(fp):
        return int(pydicom.dcmread(fp, stop_before_pixels=True).InstanceNumber)
    full_paths.sort(key=get_instance_number)
    return full_paths

def load_dicom_slice(path, img_size):
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)
    img = img - img.min()
    if img.max() > 0:
        img = img / img.max()
    img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (img_size, img_size))

def load_middle_slices(series_folder, img_size, num_slices=1):
    files = list_series_files_sorted(series_folder)
    if not files:
        return None
    mid = len(files) // 2
    half = num_slices // 2
    start = max(0, mid - half)
    end = min(len(files), start + num_slices)
    chosen = files[start:end]
    slices = [load_dicom_slice(f, img_size) for f in chosen]
    while len(slices) < num_slices:
        slices.append(slices[-1])
    return np.stack(slices, axis=-1)

def pick_series_for_study(series_df, study_id):
    candidates = series_df[
        (series_df["StudyInstanceUID"] == study_id) &
        (series_df["Anatomical_Plane"] == "Sagittal") &
        (series_df["Fluid_Sensitive"] == 1)
    ]
    if len(candidates) == 0:
        candidates = series_df[
            (series_df["StudyInstanceUID"] == study_id) &
            (series_df["Anatomical_Plane"] == "Sagittal")
        ]
    if len(candidates) == 0:
        return None
    return candidates.iloc[0]["SeriesInstanceUID"]

print("dicom_utils and select_series defined")

dicom_utils and select_series defined


In [4]:
train_augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
])

class KneeMRIDataset(Dataset):
    def __init__(self, df, series_df, img_dir, img_size=224, num_slices=3, augment=False):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.img_dir = img_dir
        self.img_size = img_size
        self.num_slices = num_slices
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]
        series_id = pick_series_for_study(self.series_df, study_id)
        if series_id is not None:
            folder = f"{self.img_dir}/{study_id}/{series_id}"
            img = load_middle_slices(folder, self.img_size, num_slices=self.num_slices)
        else:
            img = None
        if img is None:
            img = np.zeros((self.img_size, self.img_size, self.num_slices), dtype=np.uint8)
        if self.augment:
            img = train_augment(image=img)['image']
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        labels = row[LABELS].values.astype(np.float32)
        return torch.tensor(img, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

print("dataset defined")

dataset defined


In [5]:
class KneeCNN(nn.Module):
    def __init__(self, num_slices=3, num_labels=12):
        super().__init__()
        base = models.resnet18(weights=None)  # no internet needed

        old_conv = base.conv1
        new_conv = nn.Conv2d(num_slices, old_conv.out_channels,
                              kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        with torch.no_grad():
            avg_weight = old_conv.weight.mean(dim=1, keepdim=True)
            new_conv.weight[:] = avg_weight.repeat(1, num_slices, 1, 1)
        base.conv1 = new_conv
        base.fc = nn.Linear(base.fc.in_features, num_labels)
        self.backbone = base

    def forward(self, x):
        return self.backbone(x)

print("KneeCNN defined (no pretrained weights, no internet needed)")

KneeCNN defined (no pretrained weights, no internet needed)


In [6]:
def macro_auc(y_true, y_pred, labels):
    per_label, scores = {}, []
    for i, label in enumerate(labels):
        col_true, col_pred = y_true[:, i], y_pred[:, i]
        if len(np.unique(col_true)) < 2:
            continue
        score = roc_auc_score(col_true, col_pred)
        per_label[label] = score
        scores.append(score)
    overall = float(np.mean(scores)) if scores else 0.0
    return overall, per_label


def train_kfold(n_folds=5):
    study_ids = gold['StudyInstanceUID'].values
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_aucs, fold_models = [], []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(study_ids)):
        print(f"\n===== Fold {fold_idx+1}/{n_folds} =====")
        train_study_ids = study_ids[train_idx]
        val_study_ids = study_ids[val_idx]
        train_df = gold[gold['StudyInstanceUID'].isin(train_study_ids)]
        val_df = gold[gold['StudyInstanceUID'].isin(val_study_ids)]

        train_dataset = KneeMRIDataset(train_df, series_df, TRAIN_IMG_DIR, img_size=IMG_SIZE, num_slices=NUM_SLICES, augment=True)
        val_dataset = KneeMRIDataset(val_df, series_df, TRAIN_IMG_DIR, img_size=IMG_SIZE, num_slices=NUM_SLICES, augment=False)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

        model = KneeCNN(num_slices=NUM_SLICES, num_labels=len(LABELS)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.BCEWithLogitsLoss()
        best_fold_auc = 0.0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()

            model.eval()
            all_true, all_pred = [], []
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs = imgs.to(device)
                    probs = torch.sigmoid(model(imgs)).cpu().numpy()
                    all_pred.append(probs)
                    all_true.append(labels.numpy())
            all_true = np.concatenate(all_true)
            all_pred = np.concatenate(all_pred)
            val_auc, _ = macro_auc(all_true, all_pred, LABELS)
            if val_auc > best_fold_auc:
                best_fold_auc = val_auc
                torch.save(model.state_dict(), f"{MODEL_DIR}/fold_{fold_idx}_best.pt")

        print(f"Fold {fold_idx+1} best val_macro_auc: {best_fold_auc:.4f}")
        fold_aucs.append(best_fold_auc)
        fold_models.append(f"{MODEL_DIR}/fold_{fold_idx}_best.pt")

    print(f"\n===== CV RESULTS =====")
    print(f"Fold scores: {[f'{a:.4f}' for a in fold_aucs]}")
    print(f"Mean CV macro-AUC: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")
    return fold_models, fold_aucs


fold_model_paths, fold_aucs = train_kfold(n_folds=5)


===== Fold 1/5 =====
Fold 1 best val_macro_auc: 0.6287

===== Fold 2/5 =====
Fold 2 best val_macro_auc: 0.6060

===== Fold 3/5 =====
Fold 3 best val_macro_auc: 0.6324

===== Fold 4/5 =====
Fold 4 best val_macro_auc: 0.6090

===== Fold 5/5 =====
Fold 5 best val_macro_auc: 0.6932

===== CV RESULTS =====
Fold scores: ['0.6287', '0.6060', '0.6324', '0.6090', '0.6932']
Mean CV macro-AUC: 0.6339 (+/- 0.0315)


In [7]:


def generate_submission_ensemble(n_folds=5):
    test_df = pd.read_csv(TEST_CSV)
    test_series_df = pd.read_csv(TEST_SERIES_CSV)

    fold_models = []
    for i in range(n_folds):
        m = KneeCNN(num_slices=NUM_SLICES, num_labels=len(LABELS)).to(device)
        m.load_state_dict(torch.load(f"{MODEL_DIR}/fold_{i}_best.pt", map_location=device))
        m.eval()
        fold_models.append(m)

    print(f"Loaded {len(fold_models)} fold models")

    rows = []
    with torch.no_grad():
        for _, row in test_df.iterrows():
            study_id = row["StudyInstanceUID"]
            series_id = pick_series_for_study(test_series_df, study_id)

            if series_id is not None:
                folder = f"{TEST_IMG_DIR}/{study_id}/{series_id}"
                img = load_middle_slices(folder, IMG_SIZE, num_slices=NUM_SLICES)
            else:
                img = None

            if img is None:
                probs = np.full(len(LABELS), 0.1)
            else:
                img = img.astype(np.float32) / 255.0
                img = np.transpose(img, (2, 0, 1))
                tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).to(device)
                fold_probs = []
                for m in fold_models:
                    logits = m(tensor)
                    p = torch.sigmoid(logits).cpu().numpy()[0]
                    fold_probs.append(p)
                probs = np.mean(fold_probs, axis=0)

            rows.append([study_id] + probs.tolist())

    submission = pd.DataFrame(rows, columns=["StudyInstanceUID"] + LABELS)
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print(f"Saved ensembled submission.csv with {len(submission)} rows")
    return submission

submission = generate_submission_ensemble(n_folds=5)
submission

Loaded 5 fold models
Saved ensembled submission.csv with 3 rows


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.237027,0.161465,0.412060,0.356083,0.337732,0.214108,0.461177,0.420022,0.471399,0.281484,0.145363,0.217495
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.310801,0.178140,0.407586,0.407923,0.287769,0.176450,0.330960,0.381977,0.354047,0.205426,0.231679,0.249069
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.151534,0.136551,0.283096,0.358419,0.303661,0.150988,0.420443,0.118977,0.262716,0.120071,0.075703,0.164314
